# Manga Translation Benchmark Report

Visualizes `results/report.json` — compares translation methods across GLEU, BLEU, chrF, METEOR, ROUGE-L, token-length, and token-usage metrics.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REPORT = Path('results/report.json')
report = json.loads(REPORT.read_text())
cfg = report['config']
scores = report['scores']

df = pd.DataFrame(scores).T
df.index.name = 'method'
df = df.reindex(cfg['methods'])  # preserve config ordering

# flatten usage dict into columns
usage = df['usage'].apply(pd.Series).add_prefix('usage_')
df = pd.concat([df.drop(columns=['usage']), usage], axis=1)
df

## Quality metrics per method
GLEU (corpus + sentence-avg), BLEU, chrF, METEOR, ROUGE-L.

In [ ]:
metrics = ['gleu_corpus', 'gleu_sentence_avg', 'bleu', 'chrf', 'meteor', 'rouge_l']
n = len(metrics)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
for ax, metric in zip(axes.flat, metrics):
    vals = df[metric].astype(float)
    colors = ['#4C9AFF' if v < vals.max() else '#36B37E' for v in vals]
    ax.barh(vals.index, vals.values, color=colors)
    ax.set_title(metric)
    ax.invert_yaxis()
    for i, v in enumerate(vals.values):
        ax.text(v, i, f' {v:.3f}', va='center', fontsize=8)
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.suptitle(f"Translation quality by method — model={cfg['model']}, lang={cfg['target_lang']}", fontsize=13)
fig.tight_layout()
plt.show()

## Grouped comparison
All metrics on a single chart (normalized to 0-1 so they're visually comparable).

In [ ]:
norm = df[metrics].astype(float).copy()
norm = (norm - norm.min()) / (norm.max() - norm.min())

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(norm.index))
w = 0.8 / len(metrics)
for i, m in enumerate(metrics):
    ax.bar(x + i*w, norm[m].values, w, label=m)
ax.set_xticks(x + w*(len(metrics)-1)/2)
ax.set_xticklabels(norm.index, rotation=30, ha='right')
ax.set_ylabel('normalized score (min-max across methods)')
ax.set_title('Method comparison — all metrics normalized')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## Hypothesis vs reference token length
Closer to the reference average (dashed line) = better length fidelity.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ref = float(df['avg_ref_tokens'].iloc[0])
hyp = df['avg_hyp_tokens'].astype(float)
ax.bar(hyp.index, hyp.values, color='#FFAB00')
ax.axhline(ref, color='#DE350B', linestyle='--', label=f'ref avg tokens = {ref:.2f}')
for i, v in enumerate(hyp.values):
    ax.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('avg hypothesis tokens')
ax.set_title('Output length per method')
ax.set_xticklabels(hyp.index, rotation=30, ha='right')
ax.legend()
fig.tight_layout()
plt.show()

## Ranking table
Rank per metric (1 = best). Lower total = better overall.

In [ ]:
ranks = df[metrics].astype(float).rank(ascending=False).astype(int)
ranks['total'] = ranks.sum(axis=1)
ranks.sort_values('total')

## Token usage per method
Input / output / reasoning tokens consumed by the LLM. Lower = cheaper.

In [ ]:
usage_cols = ['usage_input', 'usage_output', 'usage_reasoning']
u = df[usage_cols].astype(float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# stacked bar: absolute token counts
bottom = np.zeros(len(u))
colors = {'usage_input': '#4C9AFF', 'usage_output': '#36B37E', 'usage_reasoning': '#FFAB00'}
for c in usage_cols:
    ax1.bar(u.index, u[c].values, bottom=bottom, label=c.replace('usage_', ''), color=colors[c])
    bottom += u[c].values
totals = u.sum(axis=1)
for i, v in enumerate(totals.values):
    ax1.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
ax1.set_title('Total tokens per method (stacked)')
ax1.set_ylabel('tokens')
ax1.set_xticklabels(u.index, rotation=30, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# quality-per-token: GLEU corpus / total tokens (×1000 for readability)
eff = (df['gleu_corpus'].astype(float) / totals.replace(0, np.nan)) * 1000
eff = eff.fillna(0)
ax2.bar(eff.index, eff.values, color='#6554C0')
for i, v in enumerate(eff.values):
    ax2.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax2.set_title('GLEU per 1k tokens (efficiency)')
ax2.set_ylabel('gleu_corpus × 1000 / total tokens')
ax2.set_xticklabels(eff.index, rotation=30, ha='right')
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()